# Patient Readmission Analysis & Prediction

This notebook covers the exploratory analysis and predictive modeling workflow for patient readmission. To make the notebook runnable out-of-the-box, we start by generating a realistic synthetic dataset that replicates the staging schema of our clinical records.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay

# Ensure charts output directory exists
os.makedirs('../charts', exist_ok=True)

# Set seaborn style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
print("Libraries loaded successfully.")

## 1. Data Generation (Synthetic Dataset)

We generate 1,500 patient encounters with logical clinical correlations (e.g., higher readmission rate for elderly patients, those with high prior emergency visits, or longer length of stay).

In [ ]:
np.random.seed(42)
n_samples = 1500

encounter_ids = np.arange(10001, 10001 + n_samples)
patient_ids = np.random.randint(50000, 60000, size=n_samples)

ages = np.random.choice(
    ['[0-10)', '[10-30)', '[30-50)', '[50-70)', '[70-90)'], 
    size=n_samples, 
    p=[0.05, 0.10, 0.25, 0.40, 0.20]
)

genders = np.random.choice(['Male', 'Female', 'Unknown'], size=n_samples, p=[0.48, 0.50, 0.02])
admission_types = np.random.choice(['Emergency', 'Urgent', 'Elective'], size=n_samples, p=[0.60, 0.25, 0.15])
length_of_stay = np.random.geometric(p=0.25, size=n_samples) + 1  # Mean stay ~ 5 days

diag_groups = ['Circulatory', 'Respiratory', 'Digestive', 'Diabetes', 'Injury', 'Other']
primary_diag = np.random.choice(diag_groups, size=n_samples, p=[0.30, 0.20, 0.15, 0.15, 0.10, 0.10])

num_lab_procedures = np.random.normal(loc=43, scale=15, size=n_samples).astype(int)
num_lab_procedures = np.clip(num_lab_procedures, 1, 100)

num_medications = np.random.normal(loc=16, scale=8, size=n_samples).astype(int)
num_medications = np.clip(num_medications, 1, 50)

num_emergency = np.random.poisson(lam=0.5, size=n_samples)
num_inpatient = np.random.poisson(lam=0.8, size=n_samples)

# Calculate base probability of readmission based on predictors
readmit_prob = 0.1  # base rate

# Age effect
readmit_prob += np.where(ages == '[70-90)', 0.15, 0.0)
readmit_prob += np.where(ages == '[50-70)', 0.08, 0.0)

# Length of stay and visit history effects
readmit_prob += np.clip(length_of_stay * 0.015, 0.0, 0.15)
readmit_prob += np.clip(num_emergency * 0.08, 0.0, 0.25)
readmit_prob += np.clip(num_inpatient * 0.06, 0.0, 0.20)

# Diagnosis groups
readmit_prob += np.where(primary_diag == 'Diabetes', 0.10, 0.0)
readmit_prob += np.where(primary_diag == 'Circulatory', 0.05, 0.0)

# Normalize probabilities
readmit_prob = np.clip(readmit_prob, 0.02, 0.95)
readmitted = np.random.binomial(n=1, p=readmit_prob)

# Create DataFrame
df = pd.DataFrame({
    'encounter_id': encounter_ids,
    'patient_id': patient_ids,
    'age': ages,
    'gender': genders,
    'admission_type': admission_types,
    'length_of_stay': length_of_stay,
    'primary_diagnosis_group': primary_diag,
    'num_lab_procedures': num_lab_procedures,
    'num_medications': num_medications,
    'number_emergency_visits': num_emergency,
    'number_inpatient_visits': num_inpatient,
    'readmitted': readmitted
})

df.head()

## 2. Exploratory Data Analysis (EDA)

Let's visualize the readmission distribution and relationship with clinical attributes.

In [ ]:
# Overall Readmission Rate
readmit_rate = df['readmitted'].mean() * 100
print(f"Overall 30-Day Readmission Rate: {readmit_rate:.2f}%")

# Plot 1: Readmission by Age Group
plt.figure(figsize=(8, 5))
sns.barplot(x='age', y='readmitted', data=df, errorbar=None, palette='viridis')
plt.title('Readmission Rate by Age Group')
plt.ylabel('Readmission Rate')
plt.xlabel('Age Group')
plt.savefig('../charts/readmission_by_age.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 2: Readmission by Primary Diagnosis Group
plt.figure(figsize=(10, 5))
sns.barplot(x='primary_diagnosis_group', y='readmitted', data=df, errorbar=None, palette='mako')
plt.title('Readmission Rate by Primary Diagnosis Category')
plt.ylabel('Readmission Rate')
plt.xlabel('Diagnosis Group')
plt.savefig('../charts/readmission_by_diagnosis.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Plot 3: Prior Visits Correlation
plt.figure(figsize=(8, 5))
sns.boxplot(x='readmitted', y='number_emergency_visits', data=df, palette='pastel')
plt.title('Prior Emergency Visits by Readmission Status')
plt.xlabel('Readmitted (0 = No, 1 = Yes)')
plt.ylabel('Prior Emergency Visits')
plt.savefig('../charts/readmission_by_emergency_visits.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Data Processing and Feature Engineering

We separate target and features, then construct a pipeline to scale numeric inputs and one-hot encode categorical inputs.

In [ ]:
# Drop patient IDs and encounter IDs
X = df.drop(columns=['encounter_id', 'patient_id', 'readmitted'])
y = df['readmitted']

# Define column groups
numeric_features = ['length_of_stay', 'num_lab_procedures', 'num_medications', 
                    'number_emergency_visits', 'number_inpatient_visits']
categorical_features = ['age', 'gender', 'admission_type', 'primary_diagnosis_group']

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training set shape: {X_train.shape}")
print(f"Testing set shape: {X_test.shape}")

In [ ]:
# Pipeline processors
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

## 4. Model Training & Evaluation

We compare Logistic Regression (baseline) and a Random Forest Classifier.

In [ ]:
# Model 1: Logistic Regression Pipeline
lr_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('classifier', LogisticRegression(random_state=42))])

# Model 2: Random Forest Pipeline
rf_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('classifier', RandomForestClassifier(random_state=42, n_estimators=100))])

# Fit Models
lr_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

print("Models trained successfully.")

In [ ]:
# Predict probabilities
lr_probs = lr_pipeline.predict_proba(X_test)[:, 1]
rf_probs = rf_pipeline.predict_proba(X_test)[:, 1]

# Calculate ROC-AUC
lr_auc = roc_auc_score(y_test, lr_probs)
rf_auc = roc_auc_score(y_test, rf_probs)

print(f"Logistic Regression ROC-AUC: {lr_auc:.4f}")
print(f"Random Forest ROC-AUC: {rf_auc:.4f}")

In [ ]:
# Detailed Classification Reports
lr_preds = lr_pipeline.predict(X_test)
rf_preds = rf_pipeline.predict(X_test)

print("=== Logistic Regression Classification Report ===")
print(classification_report(y_test, lr_preds))

print("\n=== Random Forest Classification Report ===")
print(classification_report(y_test, rf_preds))

In [ ]:
# Plot ROC Curves
fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_probs)
fpr_rf, tpr_rf, _ = roc_curve(y_test, rf_probs)

plt.figure(figsize=(8, 6))
plt.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC = {lr_auc:.3f})', color='dodgerblue')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {rf_auc:.3f})', color='forestgreen')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Readmission Prediction')
plt.legend(loc='lower right')
plt.savefig('../charts/roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Confusion Matrix for the Random Forest model
cm = confusion_matrix(y_test, rf_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Readmit', 'Readmit'])
disp.plot(cmap='Blues')
plt.title('Random Forest Confusion Matrix')
plt.grid(False)
plt.savefig('../charts/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Feature Importance

Let's extract and plot feature importances from the Random Forest model to understand what factors drive patient readmission.

In [ ]:
# Get feature names after column transformation
cat_encoder = rf_pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot']
encoded_cat_features = cat_encoder.get_feature_names_out(categorical_features).tolist()
all_features = numeric_features + encoded_cat_features

# Get importances
importances = rf_pipeline.named_steps['classifier'].feature_importances_
feat_imp = pd.Series(importances, index=all_features).sort_values(ascending=False)

# Plot top 10 features
plt.figure(figsize=(10, 6))
sns.barplot(x=feat_imp.head(10).values, y=feat_imp.head(10).index, palette='viridis')
plt.title('Top 10 Feature Importances (Random Forest)')
plt.xlabel('Relative Importance')
plt.ylabel('Features')
plt.savefig('../charts/feature_importances.png', dpi=300, bbox_inches='tight')
plt.show()